# 01 Data collection

**Owner:** _TBD_

Collect the raw data: proteins with sequence, Tm and organism from the Meltome Atlas, UniProt accessions and sequences, and the optimal growth temperature (OGT) of every organism.

**Reads:** downloads (Meltome via the FLIP benchmark, UniProt export), `data/raw/ogt.csv`
**Writes:** `data/interim/meltome.parquet`, `data/raw/ogt.csv` (filled in)

In [ ]:
import sys
from pathlib import Path

# Make `src` importable whether Jupyter starts in the repo root or in notebooks/.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import paths

In [ ]:
import subprocess

import pandas as pd

## 1. Meltome and UniProt

`scripts/build_meltome.py` downloads the Meltome Atlas (re-exported by the FLIP benchmark) and a UniProt
export, resolves UniProt accessions, and writes one row per protein: `id`, `sequence`, `tm`, `organism`,
`condition`. The script's docstring explains every filter. It only runs if the file does not exist yet.

In [ ]:
if not paths.MELTOME.exists():
    subprocess.run([sys.executable, str(ROOT / "scripts" / "build_meltome.py")], check=True)

meltome = pd.read_parquet(paths.MELTOME)
print(meltome.shape)
meltome.head()

In [ ]:
meltome.groupby("organism").agg(
    proteins=("id", "size"),
    tm_median=("tm", "median"),
    condition=("condition", lambda c: ", ".join(sorted(set(c)))),
).sort_values("tm_median")

## 2. Optimal growth temperature (OGT)

`data/raw/ogt.csv` lists the organisms in the dataset. Fill in `ogt` (°C) and `source` (database or paper)
for each one, for example from BacDive, TEMPURA or the literature. For animals and plants use a documented
body or growth temperature and note it in `source`.

In [ ]:
ogt = pd.read_csv(paths.OGT)
ogt

In [ ]:
missing = sorted(set(meltome["organism"]) - set(ogt.dropna(subset=["ogt"])["organism"]))
print("Organisms without OGT:", missing or "none")

## Notes and decisions

_Write down every choice made here (filters, parameters, problems) so the others can follow it._

-